<a href="https://colab.research.google.com/github/pxtroniwnl/barcelona-de-indias-time-serie/blob/main/direccion_vientoIDEAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dirección del viento — estaciones IDEAM cercanas al ROI de la laguna

**Fuente:** CSV del IDEAM para Bolívar.
**Objetivo:** seleccionar el sitio más cercano a la laguna y, dentro de él, el sensor con más observaciones.

## 1. Configuración

In [ ]:
import math
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    import folium
    from folium.plugins import Fullscreen
    HAY_FOLIUM=True
except ImportError:
    HAY_FOLIUM=False
VARIABLE='direccion_viento'; UNIDAD='grados'; NOM_ARCHIVO='Dirección_del_Viento_20260818_SOLO_BOLIVAR.csv'
# Los CSV fuente permanecen fuera del repositorio.
FUENTE_DIR=Path(r"C:\Users\Shalo\Downloads\serie_datos_laguna")
if not FUENTE_DIR.exists(): FUENTE_DIR=Path("/mnt/c/Users/Shalo/Downloads/serie_datos_laguna")
RUTA_CSV=FUENTE_DIR/NOM_ARCHIVO
FORMATO_FECHA="%Y %b %d %I:%M:%S %p"
DTYPES_TEXTO={"CodigoEstacion":"string","CodigoSensor":"string"}
COLS_TEXTO=["CodigoEstacion","CodigoSensor","NombreEstacion","Departamento","Municipio","ZonaHidrografica","DescripcionSensor","UnidadMedida"]
PASOS_SENSORES={'0104': 10}; N_ESTACIONES=2; TOL_SITIO=3
ROI_COORDS=[[-75.476052,10.517524],[-75.476117,10.518747],[-75.473158,10.519223],[-75.470516,10.525108],[-75.469572,10.524876],[-75.471686,10.518916],[-75.468394,10.517219],[-75.468952,10.516459]]
print("CSV externo:",RUTA_CSV,"| encontrado:",RUTA_CSV.exists())

## 2. Carga y exploración del CSV crudo

`df_raw` se conserva sin modificar.

In [ ]:
df_raw=pd.read_csv(RUTA_CSV,dtype=DTYPES_TEXTO,encoding="utf-8-sig",low_memory=False)
fechas_aux=pd.to_datetime(df_raw.FechaObservacion,format=FORMATO_FECHA,errors="coerce")
valores_aux=pd.to_numeric(df_raw.ValorObservado.astype("string").str.replace(",","",regex=False),errors="coerce")
print(f"{len(df_raw):,} filas × {df_raw.shape[1]} columnas")
print("Periodo:",fechas_aux.min(),"a",fechas_aux.max())
print("Fechas/valores no convertibles:",int(fechas_aux.isna().sum()),int(valores_aux.isna().sum()))
print("Sensores:",sorted(df_raw.CodigoSensor.dropna().unique().tolist()))
display(df_raw.head(3))

## 3. Limpieza

Se normaliza texto, se convierten fecha y valor, y solo se retiran filas completamente idénticas. No se imputan, interpolan, suavizan ni eliminan valores por su magnitud.

In [ ]:
def limpiar(datos):
    df=datos.copy()
    for col in COLS_TEXTO: df[col]=df[col].astype("string").str.strip().str.replace(r"\s+"," ",regex=True)
    df["FechaObservacion"]=pd.to_datetime(df.FechaObservacion,format=FORMATO_FECHA,errors="coerce")
    df["ValorObservado"]=pd.to_numeric(df.ValorObservado.astype("string").str.replace(",","",regex=False),errors="coerce")
    antes=len(df)
    df=df.drop_duplicates().sort_values(["CodigoEstacion","CodigoSensor","FechaObservacion"],kind="stable").reset_index(drop=True)
    print(f"Duplicados exactos retirados: {antes-len(df):,}")
    return df
df=limpiar(df_raw)

## 4. Catálogo, ROI y distancia

Se aplica la misma lógica Haversine del notebook guía y se agrupan códigos que representan el mismo sitio físico.

In [ ]:
def moda(s): return s.astype("string").value_counts().index[0]
base=df.dropna(subset=["FechaObservacion"])
ultimo=(base.sort_values("FechaObservacion",kind="stable").drop_duplicates("CodigoEstacion",keep="last").set_index("CodigoEstacion")[["Latitud","Longitud"]])
catalogo=base.groupby("CodigoEstacion").agg(NombreEstacion=("NombreEstacion",moda),Municipio=("Municipio",moda),n_obs=("ValorObservado","count"),inicio=("FechaObservacion","min"),fin=("FechaObservacion","max")).join(ultimo).reset_index()
_lons=[x[0] for x in ROI_COORDS]; _lats=[x[1] for x in ROI_COORDS]; CX=(min(_lons)+max(_lons))/2; CY=(min(_lats)+max(_lats))/2
def haversine(lat,lon):
    lat=np.radians(np.asarray(lat,float)); lon=np.radians(np.asarray(lon,float)); a=np.sin((math.radians(CY)-lat)/2)**2+np.cos(lat)*math.cos(math.radians(CY))*np.sin((math.radians(CX)-lon)/2)**2; return 2*6371.0088*np.arcsin(np.sqrt(a))
estaciones=catalogo.copy(); estaciones["dist_km"]=haversine(estaciones.Latitud,estaciones.Longitud).round(2); estaciones["_lat"]=estaciones.Latitud.round(TOL_SITIO); estaciones["_lon"]=estaciones.Longitud.round(TOL_SITIO)
grupos=estaciones.groupby(["_lat","_lon"]).CodigoEstacion.agg(list).rename("codigos_del_sitio")
sitios=(estaciones.sort_values("dist_km",kind="stable").drop_duplicates(["_lat","_lon"]).join(grupos,on=["_lat","_lon"]).drop(columns=["_lat","_lon"]).reset_index(drop=True))
print(f"Centro del ROI: lat={CY:.6f}, lon={CX:.6f}")
display(sitios[["NombreEstacion","CodigoEstacion","codigos_del_sitio","dist_km","n_obs"]].head(6))

## 5. Mapa y comparación de los dos sitios más cercanos

In [ ]:
sitios_comparados=sitios.head(N_ESTACIONES)
filas=[]
for _,sitio in sitios_comparados.iterrows():
    sub=df[df.CodigoEstacion.isin(sitio.codigos_del_sitio)]
    filas.append({"sitio":sitio.NombreEstacion,"dist_km":sitio.dist_km,"codigos":", ".join(sitio.codigos_del_sitio),"sensores":", ".join(sorted(sub.CodigoSensor.dropna().unique().tolist())),"observaciones":int(sub.ValorObservado.count()),"inicio":sub.FechaObservacion.min(),"fin":sub.FechaObservacion.max()})
display(pd.DataFrame(filas))
if HAY_FOLIUM:
    mapa=folium.Map(location=[CY,CX],zoom_start=11,tiles="CartoDB positron"); Fullscreen().add_to(mapa)
    folium.Polygon([(lat,lon) for lon,lat in ROI_COORDS],color="cyan",fill=True,fill_opacity=.25).add_to(mapa)
    for _,r in sitios_comparados.iterrows(): folium.Marker([r.Latitud,r.Longitud],popup=f"{r.NombreEstacion}: {r.dist_km:.2f} km").add_to(mapa)
    display(mapa)

## 6. Comparación de sensores y selección

La tabla muestra cantidad de observaciones y cobertura. La selección usa la mayor cantidad de observaciones en el sitio más cercano.

In [ ]:
sitio_cercano=sitios.iloc[0]; codigos=sitio_cercano.codigos_del_sitio
candidatas=df[df.CodigoEstacion.isin(codigos)].dropna(subset=["FechaObservacion","ValorObservado"])
filas=[]
for sensor,datos in candidatas.groupby("CodigoSensor"):
    fechas=datos.FechaObservacion.drop_duplicates().sort_values(); paso=PASOS_SENSORES[str(sensor)]; esperadas=int((fechas.iloc[-1]-fechas.iloc[0]).total_seconds()//(paso*60))+1; presentes=int((((fechas-fechas.iloc[0]).dt.total_seconds()%(paso*60))==0).sum())
    filas.append({"CodigoSensor":sensor,"codigos":", ".join(sorted(datos.CodigoEstacion.unique().tolist())),"inicio":fechas.iloc[0],"fin":fechas.iloc[-1],"frecuencia_min":paso,"n_observaciones":len(datos),"esperadas":esperadas,"faltantes":esperadas-presentes,"cobertura_pct":round(100*presentes/esperadas,2)})
comparacion_sensores=pd.DataFrame(filas).sort_values(["n_observaciones","cobertura_pct","CodigoSensor"],ascending=[False,False,True],kind="stable").reset_index(drop=True)
display(comparacion_sensores)
SENSOR_SELECCIONADO=comparacion_sensores.loc[0,"CodigoSensor"]
serie_nativa=candidatas[candidatas.CodigoSensor==SENSOR_SELECCIONADO].copy(); serie_nativa["Estacion"]=sitio_cercano.NombreEstacion; serie_nativa=serie_nativa.sort_values(["FechaObservacion","CodigoEstacion"],kind="stable").reset_index(drop=True)
print("Sitio:",sitio_cercano.NombreEstacion,"| sensor:",SENSOR_SELECCIONADO,"| mediciones:",len(serie_nativa))
assert SENSOR_SELECCIONADO=='0104'

## 7. Diagnóstico, serie horaria y visualización

In [ ]:
paso=PASOS_SENSORES[str(SENSOR_SELECCIONADO)]; fechas=serie_nativa.FechaObservacion.drop_duplicates().sort_values(); esperadas=int((fechas.iloc[-1]-fechas.iloc[0]).total_seconds()//(paso*60))+1
display(pd.DataFrame([{"Estacion":sitio_cercano.NombreEstacion,"CodigoSensor":SENSOR_SELECCIONADO,"observaciones":len(serie_nativa),"inicio":fechas.iloc[0],"fin":fechas.iloc[-1],"frecuencia_min":paso,"cobertura_pct":round(100*len(fechas)/esperadas,2)}]))
def circular(s):
    a=np.radians(s.to_numpy(float)); return float(np.degrees(np.arctan2(np.sin(a).mean(),np.cos(a).mean()))%360)
func=circular if VARIABLE=="direccion_viento" else "mean"
horaria=serie_nativa.groupby(pd.Grouper(key="FechaObservacion",freq="h")).ValorObservado.agg(valor_horario=func,n_crudos="size").dropna().reset_index()
fig,ax=plt.subplots(figsize=(15,4)); ax.plot(horaria.FechaObservacion,horaria.valor_horario,lw=.45,color="#2E8B8B"); ax.set(title=f"{VARIABLE.replace('_',' ').title()} — {sitio_cercano.NombreEstacion}",xlabel="Fecha",ylabel=UNIDAD); ax.grid(alpha=.25); plt.tight_layout(); plt.show()

## 8. Exportación de la serie nativa seleccionada

In [ ]:
EXPORT_DIR=Path("data/ideam"); EXPORT_DIR.mkdir(parents=True,exist_ok=True)
RUTA_EXPORT=EXPORT_DIR/'direccion_viento_rafael_nunez.csv'
exportar=serie_nativa[["Estacion","CodigoEstacion","FechaObservacion","ValorObservado"]]
exportar.to_csv(RUTA_EXPORT,index=False,date_format="%Y-%m-%d %H:%M:%S")
print("Exportado:",RUTA_EXPORT.resolve(),"| filas:",len(exportar)); display(exportar.head())